In [1]:
%pip install transformers bitsandbytes accelerate torch kernels

  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.4.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

CUDA available: True
GPU name: NVIDIA H200 NVL
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from larg

In [3]:
import os

folder_path = "tables/"

folder_path_LLM_statements = "GPT/b.LLM_Inferences"

folder_path_python_code = "GPT/c.checking_statements"

folder_path_python_output_checking_statements = "GPT/d.checking_statements_output"

In [4]:
#initalizing the model with 4 bit quantization

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "openai/gpt-oss-120b"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16, 
    low_cpu_mem_usage=True,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/615 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

In [5]:
from transformers import pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [11]:
import re

# List all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
csv_files.sort()  # optional: ensure consistent order

for csv_file in csv_files:
    full_path = os.path.join(folder_path, csv_file)

    # Read the table
    with open(full_path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{csv_file} is empty, skipping")
        continue

    # Extract header and rows
    header = lines[0]
    rows = lines[1:]

    print(f"Processing {csv_file}: {len(rows)} rows")

    prompt1 = [
    {
        "role": "system",
        "content": """You are an expert data analyst and logician.

You may think silently, but your visible output must be ONLY valid JSON.

Return exactly one JSON object with this schema:
{"statements": ["...", "..."]}

Rules:
- 5 to 10 statements
- each statement must be factually true from the table
- non-trivial and high-information
- natural language
- no markdown
- no explanation
- no preamble
- no trailing text
- do not wrap the JSON in code fences
"""
    },
    {
        "role": "user",
        "content": f"""
        Your task is to generate 5-10 natural language statements that describe patterns, relationships, and notable observations in this data.

REQUIREMENTS:
1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.
2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21-43" or "high-income individuals"). Avoid arbitrary correlations.
3. **Clarity**: Use natural language that is easy to understand. Avoid overly complex nested conditions.
4. **Variety**: Mix different types of statements:
   - Universal claims: "All X satisfy property Y"
   - Existential claims: "There exists at least one X that satisfies Y"
   - Majority/frequency claims: "Most X have property Y"

AVOID:
- Overly specific conditions that apply to only 1-2 individuals
- Redundant statements that express the same fact in different ways
- Statements too complex to parse (keep conditions to 2-3 attributes max)
- Probabilistic language (e.g., "likely", "probably") unless you have strong statistical support

Here is the data:
{lines}

Generate your statements below, one per line."""
    },
]
    
    #Generate the statements necessary
    generation = generator(
    prompt1,
    do_sample=False,
    temperature=1.0,
    top_p=1,
    max_new_tokens=10000,
    eos_token_id=tokenizer.eos_token_id)
    print(f"Generation: {generation[0]['generated_text']}")
    # Get the assistant message from generated_text
    statements_LLM_output = generation[-1]['generated_text'][-1]  # last item
    clean_statements_LLM_output_text = statements_LLM_output['content']  # this is your CSV string
    statements_LLM_output_text = re.sub(r"<think>.*?</think>", "", clean_statements_LLM_output_text, flags=re.DOTALL)
    # Preview
    print(statements_LLM_output_text[1000:2000])
    #save the output of the LLM generated tasks
    file_name_LLM = f"LLM_statements_{csv_file[:-4]}.txt"

    folder_path_LLM_statements = "GPT/b.LLM_Inferences"

    full_path_LLM_statements = os.path.join(folder_path_LLM_statements, file_name_LLM)


    with open(full_path_LLM_statements, "w", encoding="utf-8") as f:
      f.write(statements_LLM_output_text)
    print(f"Saved {file_name_LLM}.")


Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing table_0.csv: 15 rows


Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician.\n\nYou may think silently, but your visible output must be ONLY valid JSON.\n\nReturn exactly one JSON object with this schema:\n{"statements": ["...", "..."]}\n\nRules:\n- 5 to 10 statements\n- each statement must be factually true from the table\n- non-trivial and high-information\n- natural language\n- no markdown\n- no explanation\n- no preamble\n- no trailing text\n- do not wrap the JSON in code fences\n'}, {'role': 'user', 'content': '\n        Your task is to generate 5-10 natural language statements that describe patterns, relationships, and notable observations in this data.\n\nREQUIREMENTS:\n1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.\n2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21-43" or "high-income individuals"). 

Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician.\n\nYou may think silently, but your visible output must be ONLY valid JSON.\n\nReturn exactly one JSON object with this schema:\n{"statements": ["...", "..."]}\n\nRules:\n- 5 to 10 statements\n- each statement must be factually true from the table\n- non-trivial and high-information\n- natural language\n- no markdown\n- no explanation\n- no preamble\n- no trailing text\n- do not wrap the JSON in code fences\n'}, {'role': 'user', 'content': '\n        Your task is to generate 5-10 natural language statements that describe patterns, relationships, and notable observations in this data.\n\nREQUIREMENTS:\n1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.\n2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21-43" or "high-income individuals"). 

Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician.\n\nYou may think silently, but your visible output must be ONLY valid JSON.\n\nReturn exactly one JSON object with this schema:\n{"statements": ["...", "..."]}\n\nRules:\n- 5 to 10 statements\n- each statement must be factually true from the table\n- non-trivial and high-information\n- natural language\n- no markdown\n- no explanation\n- no preamble\n- no trailing text\n- do not wrap the JSON in code fences\n'}, {'role': 'user', 'content': '\n        Your task is to generate 5-10 natural language statements that describe patterns, relationships, and notable observations in this data.\n\nREQUIREMENTS:\n1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.\n2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21-43" or "high-income individuals"). 

In [14]:
import os
import re
import json
import ast
import subprocess
import pandas as pd

# helper to extract code fences if present
def extract_python_code(raw: str) -> str:
    """
    Strip markdown fences if present (```python ... ``` or ``` ... ```).
    Falls back to returning the full string if no fences are found.
    """
    match = re.search(r"```(?:python)?\s*\n(.*?)```", raw, re.DOTALL)
    if match:
        return match.group(1).strip()
    return raw.strip()

def extract_statements_from_txt(raw: str) -> list[str]:
    """
    Extract the statements list from messy model output like:
    assistantfinal{"statements":[...]}
    """
    raw = raw.strip()

    match = re.search(r'(\{\s*"statements"\s*:\s*\[.*?\]\s*\})', raw, re.DOTALL)
    if not match:
        raise ValueError("Could not find a JSON object with a 'statements' field.")

    obj_text = match.group(1)

    try:
        obj = json.loads(obj_text)
    except Exception:
        obj = ast.literal_eval(obj_text)

    statements = obj["statements"]
    if not isinstance(statements, list):
        raise ValueError("'statements' is not a list.")

    return [str(s).strip() for s in statements if str(s).strip()]

def extract_real_python(raw: str) -> str:
    """
    Remove model thinking and keep only the actual Python code.
    Priority:
    1. Content after 'assistantfinal'
    2. Start from first 'import pandas as pd'
    3. Fallback to first import/from line
    """
    raw = raw.strip()

    # Prefer everything after assistantfinal
    m = re.search(r"assistantfinal\s*(.*)$", raw, re.DOTALL)
    if m:
        candidate = m.group(1).strip()
    else:
        candidate = raw

    # Remove markdown fences if any
    candidate = extract_python_code(candidate)

    # Best anchor: code starts at import pandas as pd
    m = re.search(r"(?ms)^import pandas as pd\b.*$", candidate)
    if m:
        return m.group(0).strip()

    # Fallback: start at first import/from line
    m = re.search(r"(?ms)^(?:import|from)\s+.*$", candidate)
    if m:
        return m.group(0).strip()

    return candidate.strip()

LLM_statement_text_files = sorted(
    f for f in os.listdir(folder_path_LLM_statements) if f.endswith(".txt")
)

csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

for LLM_statement_text_file in LLM_statement_text_files:
    full_path_stmt = os.path.join(folder_path_LLM_statements, LLM_statement_text_file)

    with open(full_path_stmt, "r", encoding="utf-8") as f:
        raw_stmt_text = f.read()

    if not raw_stmt_text.strip():
        print(f"{LLM_statement_text_file} is empty, skipping")
        continue

    try:
        statements = extract_statements_from_txt(raw_stmt_text)
    except Exception as e:
        print(f"{LLM_statement_text_file}: failed to parse statements -> {e}")
        continue

    print(f"\nProcessing {LLM_statement_text_file}.")
    print(f"Parsed {len(statements)} statements.")

    statements_text = "\n".join(
        f"{i+1}. {stmt}" for i, stmt in enumerate(statements)
    )

    matched_csv = None
    for csv_file in csv_files:
        if csv_file[:-4] in LLM_statement_text_file:
            matched_csv = csv_file
            break

    if matched_csv is None:
        print(f"No matching CSV found for {LLM_statement_text_file}, skipping.")
        continue

    full_csv_path = os.path.join(folder_path, matched_csv)
    df = pd.read_csv(full_csv_path)

    prompt2 = [
        {"role": "system", "content": "You are an expert data analyst."},
        {"role": "user", "content": f"""Do NOT repeat the instructions or the code provided.
Only output the requested Python code. Do NOT wrap the code in markdown fences.

Here's your task: Given the following statements:

{statements_text}

and the header names of the table {list(df.columns)}, write a python code (using pandas package)
that checks whether each statement is True or False and prints a justification.

The CSV is already located at: "{full_csv_path}" — hardcode this path directly in the script (no sys.argv).
It should also convert any turn numbers stored as strings into integers.
Everything you output must be valid, immediately runnable Python with no markdown or commentary outside comments.

Here is an example structure to follow:
import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\\nStatement {{statement_no}}: {{status}}")
    print(f"  - {{description}}")
    print(f"  - Explanation: {{explanation}}")

def stmt_1(df: pd.DataFrame):
    \"\"\"1. For all individuals, if the person is a woman, then her age is between 21 and 43.\"\"\"
    women = df[df["gender"] == "F"]
    condition = women["age"].between(21, 43, inclusive="both")
    truth = condition.all()
    if truth:
        expl = f"All {{len(women)}} women are aged 21–43."
    else:
        viol = women[~condition]
        expl = f"{{len(viol)}} women violate the rule (ages: {{', '.join(map(str, viol['age'].tolist()))}})."
    return truth, expl

def main():
    df = pd.read_csv("{full_csv_path}")
    checks = [(1, stmt_1)]  # extend for all statements
    for num, func in checks:
        truth, explanation = func(df)
        print_result(num, func.__doc__.strip(), truth, explanation)

if __name__ == "__main__":
    main()"""}
    ]

    # ── Generate ───────────────────────────────────────────────────────────
    generation = generator(
        prompt2,
        do_sample=False,
        temperature=1.0,
        top_p=1,
        max_new_tokens=100000,
        eos_token_id=tokenizer.eos_token_id,
    )

    python_LLM_output = generation[-1]["generated_text"][-1]
    raw_code = python_LLM_output["content"]

    # ── Clean: keep only actual python code ───────────────────────────────
    python_code = extract_real_python(raw_code)

    # ── Save ───────────────────────────────────────────────────────────────
    python_file_name_LLM = f"python_code_{matched_csv[:-4]}.py"
    full_path_py = os.path.join(folder_path_python_code, python_file_name_LLM)

    with open(full_path_py, "w", encoding="utf-8") as f:
        f.write(python_code)
    print(f"Saved {python_file_name_LLM}")

    # ── Execute automatically ──────────────────────────────────────────────
    print(f"Running {python_file_name_LLM}...")
    result = subprocess.run(
        ["python3", full_path_py],
        capture_output=True,
        text=True,
    )

    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"[ERROR] Script exited with code {result.returncode}")
        print(result.stderr)
    else:
        print(f"[OK] {python_file_name_LLM} completed successfully.")

    results_file_name = f"validation_gpt_inferences_{matched_csv[:-4]}.txt"
    full_path_results_file = os.path.join(
        folder_path_python_output_checking_statements,
        results_file_name
    )

    with open(full_path_results_file, "w", encoding="utf-8") as f:
        f.write(result.stdout)
        if result.returncode != 0:
            f.write(f"\n[ERROR] Script exited with code {result.returncode}\n")
            f.write(result.stderr)

    print(f"Saved {results_file_name}")

Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Processing LLM_statements_table_0.txt.
Parsed 8 statements.
Saved python_code_table_0.py
Running python_code_table_0.py...


Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Statement 1: TRUE
  - All students who are club members have attendance rates above 89%.
  - Explanation: All 8 club members have attendance > 89%.

Statement 2: TRUE
  - All 12th‑grade students scored at least 78 on the test.
  - Explanation: All 3 12th‑grade students scored ≥ 78.

Statement 3: TRUE
  - All students who study more than 9 hours per week scored 90 or higher on the test.
  - Explanation: All 4 students studying >9 hrs/week scored ≥ 90.

Statement 4: TRUE
  - Most 10th‑grade students are not members of a club.
  - Explanation: 3 out of 4 (≈75.0%) 10th‑graders are not club members.

Statement 5: TRUE
  - The student with the highest attendance rate (99.1%) also achieved the highest test score (95).
  - Explanation: Highest attendance is 99.1% and highest test score is 95, both belonging to the same student.

Statement 6: FALSE
  - All students with attendance rates of 95% or higher scored at least 88 on the test.
  - Explanation: 1 students with attendance ≥95% scored bel

Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved python_code_table_1.py
Running python_code_table_1.py...
[ERROR] Script exited with code 1
  File "/home/ayushs13/inference_generation/GPT/c.checking_statements/python_code_table_1.py", line 23
    expl = f"{len(viol)} hypertension patient(s) are non‑smokers (IDs: {', '.join(viol['patient_id']))})."
                                                                                                     ^
SyntaxError: f-string: unmatched ')'

Saved validation_gpt_inferences_table_1.txt

Processing LLM_statements_table_2.txt.
Parsed 8 statements.
Saved python_code_table_2.py
Running python_code_table_2.py...


Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Statement 1: TRUE
  - All north region stores have customer satisfaction scores of at least 4.3.
  - Explanation: All 4 north stores meet the threshold.

Statement 2: TRUE
  - All south region stores have customer satisfaction of 4.0 or higher.
  - Explanation: All 4 south stores meet the threshold.

Statement 3: TRUE
  - The store with the highest staff count (B009) also records the highest monthly sales.
  - Explanation: Store B009 has the highest staff count (23) and also the highest sales (165.8k).

Statement 4: TRUE
  - Stores with an average basket size above 62 have customer satisfaction of at least 4.5.
  - Explanation: All 3 stores with basket >62 meet the satisfaction threshold.

Statement 5: TRUE
  - There exists a west region store with customer satisfaction below 4.0.
  - Explanation: Found 3 west store(s) below 4.0 (store_id: B004, B008, B012).

Statement 6: TRUE
  - Most stores with staff counts of 20 or more have monthly sales exceeding 130,000.
  - Explanation: 4 out 